# Precompute session-level trial metadata directly from ONE

Fetches `choice`, `feedbackType` (outcome), `probabilityLeft` (block), and `goCueTrigger_times`
straight from ONE for every LDA-tracked session (the same session set used by
`firing_rate_computation.ipynb`), and saves one consolidated file.

This replaces dependence on `4_mice/all_trials_04-05-2026`, which was generated by the now-deprecated
`0_syllable_trial_type.ipynb` pipeline (predates the no-go/choice fixes made to `segmentation_functions.py`
and silently drops every no-go trial). Fetching live from ONE avoids inheriting any of that pipeline's bugs.

`goCueTrigger_times` is included specifically so that consuming notebooks (e.g. `noise_corr_psth_conditions.ipynb`)
can validate the `trial_id` join against each firing-rate file's own `event_time` column before trusting it,
rather than assuming `trial_id` alignment silently.

`side`/`contrast` are intentionally NOT duplicated here: `firing_rate_computation.ipynb` already computes them
correctly from `contrastLeft`/`contrastRight` via ONE (see its `condition` column), so re-deriving them here
would be redundant.

choice/feedback string convention (matches `segmentation_functions.py`'s `align_bin_design_matrix`,
verified against a known trial - see notebook discussion):
- choice:   1 -> 'right', -1 -> 'left', 0 -> 'no_go'
- feedback: 1 -> 'correct', -1 -> 'incorrect'

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
from one.api import ONE
import brainwidemap

one = ONE()
prefix = '/Users/ineslaranjeira/Documents/Repositories/'

## Load the LDA session list (same source as `firing_rate_computation.ipynb`)

In [ ]:
bwm_df = brainwidemap.bwm_query()
n_sessions = bwm_df['eid'].unique().shape[0]
n_insertions = bwm_df['pid'].unique().shape[0]
print(f'{n_sessions} sessions with {n_insertions} individual neuropixel recordings')

data_path = prefix + 'representation_learning_variability/paper-individuality/clustering/'
lda = pd.read_pickle(data_path + 'mouse_LDA_5_bins_cut19-06-2026')
lda = lda.rename(columns={0: 'lda_1'})

lda_eid = lda.loc[lda['session'].isin(list(bwm_df.eid)), 'session']
sessions = sorted(lda_eid.unique())
print(f'{len(sessions)} unique LDA-matched sessions to fetch')

## Fetch trial metadata per session

In [ ]:
CHOICE_MAP = {1.0: 'right', -1.0: 'left', 0.0: 'no_go'}
FEEDBACK_MAP = {1.0: 'correct', -1.0: 'incorrect'}

records = []
failed_sessions = {}

for i, session in enumerate(sessions):
    try:
        trials = one.load_object(session, 'trials', collection='alf')
    except Exception as e:
        print(f'  Could not fetch trials for {session}: {e}')
        failed_sessions[session] = str(e)
        continue

    n = len(trials['choice'])
    df = pd.DataFrame({
        'session': session,
        'trial_id': np.arange(n),
        'choice': pd.Series(trials['choice']).map(CHOICE_MAP),
        'feedback': pd.Series(trials['feedbackType']).map(FEEDBACK_MAP),
        'block': np.asarray(trials['probabilityLeft']),
        'goCueTrigger_times': np.asarray(trials['goCueTrigger_times']),
    })
    records.append(df)
    if (i + 1) % 25 == 0:
        print(f'  {i+1}/{len(sessions)} sessions fetched...')

session_trial_meta = pd.concat(records, ignore_index=True)
print(f'\nFetched {len(session_trial_meta)} trial rows across {session_trial_meta["session"].nunique()} sessions')
if failed_sessions:
    print(f'{len(failed_sessions)} sessions failed to fetch: {list(failed_sessions.keys())}')

## Save

In [ ]:
save_path = prefix + 'representation_learning_variability/paper-individuality/4_mice/'
current_date = datetime.now().strftime('%d-%m-%Y')
filename = f'session_trial_meta_{current_date}'
session_trial_meta.to_parquet(save_path + filename, compression='gzip')
print(f'Saved to {save_path + filename}')